# 03 — Consumer: DuckDB & Pandas (No Warehouse!)

This notebook is the **consumer** side of the demo. It reads a shared Delta table
using only open-source tools — **no Databricks SQL warehouse, no JDBC, no cluster**.

All you need is:
- `delta-sharing` Python library (reads the open Delta Sharing protocol)
- `duckdb` (in-process SQL engine)
- `pandas` (data manipulation)
- The `share_profile.json` file from the provider

In [1]:
import delta_sharing
import duckdb
import pandas as pd

PROFILE_PATH = "share_profile.json"

## Step 1 — Discover what's been shared with us

Using the `delta_sharing` library to list available shares and tables.
This is **all running locally** — no Databricks compute involved.

In [2]:
# Create a sharing client from the profile
client = delta_sharing.SharingClient(PROFILE_PATH)

# List all shares available to this recipient
shares = client.list_shares()
print("Available shares:")
for share in shares:
    print(f"  - {share.name}")

# List all tables in each share
print("\nAvailable tables:")
all_tables = client.list_all_tables()
for table in all_tables:
    print(f"  - {table.share}.{table.schema}.{table.name}")

Available shares:
  - nfl_combine_share

Available tables:
  - nfl_combine_share.nfl_combine.combine_results


## Step 2 — Load into pandas (one line!)

`delta_sharing.load_as_pandas()` reads the shared table directly into a DataFrame.
The data is fetched via the Delta Sharing REST protocol — **zero warehouse cost**.

In [3]:
# The table URL format is: <profile_path>#<share>.<schema>.<table>
table_url = f"{PROFILE_PATH}#nfl_combine_share.nfl_combine.combine_results"

# Load the entire shared table into a pandas DataFrame — no warehouse needed!
df = delta_sharing.load_as_pandas(table_url)

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
df.head(10)

Loaded 300 rows, 12 columns
Columns: ['player_name', 'position', 'college', 'height_in', 'weight_lbs', 'forty_yard_dash', 'bench_press_reps', 'vertical_jump_in', 'broad_jump_in', 'three_cone_drill', 'shuttle_20yd', 'draft_year']


,player_name,position,college,height_in,weight_lbs,forty_yard_dash,bench_press_reps,vertical_jump_in,broad_jump_in,three_cone_drill,shuttle_20yd,draft_year
0,Alexander Hill,CB,Texas,69,203,4.60,17,27.4,111,7.38,4.52,2023
1,Noah Rhodes,LB,Kentucky,72,226,4.40,17,33.6,108,7.23,4.40,2025
2,Brandon Henderson,DT,Baylor,74,318,4.95,35,39.8,132,7.55,4.39,2024
3,Daniel Wagner,OT,Oregon,77,316,4.41,22,26.6,119,7.28,4.46,2025
4,Cristian Santos,DE,Texas,77,252,4.91,30,35.5,119,7.25,4.39,2023
5,Daniel Lawrence,CB,Texas A&M,71,182,5.24,13,31.5,122,7.33,4.16,2024
6,Aaron Shaffer,OG,Tennessee,76,299,4.97,15,34.1,115,6.71,4.17,2025
7,Gerald Moore,S,Texas A&M,72,196,4.55,11,38.7,120,6.85,4.05,2025
8,George Davis,S,UCLA,71,215,4.85,30,32.8,116,6.68,4.42,2025
9,Ryan Munoz,OT,Kentucky,80,320,4.70,14,33.7,110,7.48,4.50,2023


## Step 3 — Query with DuckDB (SELECT * and more!)

DuckDB can query pandas DataFrames directly with SQL. This gives you a full SQL
engine on top of the shared data — **entirely in-process, no server needed**.

In [4]:
# DuckDB can query pandas DataFrames directly by name!
con = duckdb.connect()

# SELECT * — the simplest possible query, no warehouse needed
print("=== SELECT * FROM combine_results (first 10 rows) ===\n")
con.sql("SELECT * FROM df LIMIT 10").show()

=== SELECT * FROM combine_results (first 10 rows) ===

┌───────────────────┬──────────┬───────────┬───────────┬────────────┬─────────────────┬──────────────────┬──────────────────┬───────────────┬──────────────────┬──────────────┬────────────┐
│    player_name    │ position │  college  │ height_in │ weight_lbs │ forty_yard_dash │ bench_press_reps │ vertical_jump_in │ broad_jump_in │ three_cone_drill │ shuttle_20yd │ draft_year │
│      varchar      │ varchar  │  varchar  │   int64   │   int64    │     double      │      int64       │      double      │     int64     │      double      │    double    │   int64    │
├───────────────────┼──────────┼───────────┼───────────┼────────────┼─────────────────┼──────────────────┼──────────────────┼───────────────┼──────────────────┼──────────────┼────────────┤
│ Alexander Hill    │ CB       │ Texas     │        69 │        203 │             4.6 │               17 │             27.4 │           111 │             7.38 │         4.52 │       2023 │


In [5]:
# Aggregate query — fastest 40-yard dash by position
print("=== Fastest 40-yard dash by position ===\n")
con.sql("""
    SELECT
        position,
        COUNT(*)                          AS players,
        ROUND(MIN(forty_yard_dash), 2)    AS fastest_40,
        ROUND(AVG(forty_yard_dash), 2)    AS avg_40,
        ROUND(AVG(bench_press_reps), 1)   AS avg_bench
    FROM df
    GROUP BY position
    ORDER BY fastest_40
""").show()

=== Fastest 40-yard dash by position ===

┌──────────┬─────────┬────────────┬────────┬───────────┐
│ position │ players │ fastest_40 │ avg_40 │ avg_bench │
│ varchar  │  int64  │   double   │ double │  double   │
├──────────┼─────────┼────────────┼────────┼───────────┤
│ K        │      21 │        4.3 │   4.89 │      21.0 │
│ QB       │      21 │        4.3 │    4.9 │      20.3 │
│ DE       │      23 │        4.3 │   4.87 │      25.4 │
│ CB       │      21 │        4.3 │   4.78 │      20.8 │
│ P        │      30 │       4.31 │   4.76 │      21.9 │
│ C        │      23 │       4.32 │    4.8 │      23.3 │
│ OT       │      23 │       4.32 │   4.86 │      22.3 │
│ OG       │      20 │       4.33 │   4.75 │      26.3 │
│ LB       │      33 │       4.33 │   4.82 │      20.8 │
│ WR       │      24 │       4.35 │   4.79 │      24.0 │
│ RB       │      15 │       4.36 │   4.89 │      21.8 │
│ S        │      20 │       4.39 │   4.86 │      19.8 │
│ TE       │      13 │       4.41 │   4.79 │  

In [6]:
# Filter query — top WR prospects with sub-4.5 forty
print("=== Elite WR prospects (sub-4.5 forty) ===\n")
con.sql("""
    SELECT player_name, college, draft_year,
           forty_yard_dash, vertical_jump_in, broad_jump_in
    FROM df
    WHERE position = 'WR' AND forty_yard_dash < 4.5
    ORDER BY forty_yard_dash
""").show()

=== Elite WR prospects (sub-4.5 forty) ===

┌───────────────┬────────────┬────────────┬─────────────────┬──────────────────┬───────────────┐
│  player_name  │  college   │ draft_year │ forty_yard_dash │ vertical_jump_in │ broad_jump_in │
│    varchar    │  varchar   │   int64    │     double      │      double      │     int64     │
├───────────────┼────────────┼────────────┼─────────────────┼──────────────────┼───────────────┤
│ James Lewis   │ Utah       │       2023 │            4.35 │             35.5 │           121 │
│ Dustin Nelson │ Florida    │       2023 │            4.45 │             25.9 │           123 │
│ Matthew Moore │ Penn State │       2023 │            4.45 │             39.3 │           127 │
└───────────────┴────────────┴────────────┴─────────────────┴──────────────────┴───────────────┘



## Step 4 — Pandas analytics on the shared data

Once loaded, it's just a regular pandas DataFrame — use any pandas operation.

In [7]:
# Position breakdown
print("=== Players per position ===")
print(df["position"].value_counts().to_string())

print("\n=== Average combine metrics by draft year ===")
print(
    df.groupby("draft_year")[["forty_yard_dash", "bench_press_reps", "vertical_jump_in"]]
    .mean()
    .round(2)
    .to_string()
)

=== Players per position ===
position
LB    33
P     30
WR    24
OT    23
DE    23
C     23
CB    21
K     21
QB    21
OG    20
S     20
RB    15
DT    13
TE    13

=== Average combine metrics by draft year ===
            forty_yard_dash  bench_press_reps  vertical_jump_in
draft_year                                                     
2023                   4.80             22.19             33.94
2024                   4.88             23.23             33.64
2025                   4.82             22.44             33.23


In [8]:
# Composite "athleticism score" — just pandas, no warehouse
df["athleticism_score"] = (
    (1 / df["forty_yard_dash"]) * 100
    + df["vertical_jump_in"]
    + df["broad_jump_in"] / 3
    + df["bench_press_reps"]
).round(1)

top_athletes = df.nlargest(15, "athleticism_score")[
    ["player_name", "position", "college", "draft_year", "athleticism_score",
     "forty_yard_dash", "vertical_jump_in", "bench_press_reps"]
]

print("=== Top 15 athletes by composite score ===")
top_athletes

=== Top 15 athletes by composite score ===


,player_name,position,college,draft_year,athleticism_score,forty_yard_dash,vertical_jump_in,bench_press_reps
189,Jeffrey Simon,OT,Texas,2023,139.6,4.73,41.8,35
2,Brandon Henderson,DT,Baylor,2024,139.0,4.95,39.8,35
247,Isaiah Avila,LB,North Carolina,2024,137.9,5.00,41.9,35
145,Travis Vance,DT,Pittsburgh,2025,136.2,5.04,42.0,33
71,Zachary Peters,WR,Wisconsin,2023,136.0,4.91,41.0,32
218,Darius Mills,OG,Pittsburgh,2023,135.3,4.71,41.4,31
262,Joshua Reed,S,Texas A&M,2023,135.1,4.73,36.6,34
34,Derek Davidson,WR,Kentucky,2023,134.4,4.64,41.5,35
77,David Wilson,S,Colorado,2024,134.3,5.32,38.5,35
143,Ryan Bryant,DT,Pittsburgh,2025,134.3,5.10,41.4,31


## Step 5 — DuckDB: Write results to local Parquet or CSV

DuckDB can also export query results — the consumer can save locally without
ever touching Databricks infrastructure.

In [9]:
# Export a DuckDB query result to local Parquet
con.sql("""
    COPY (
        SELECT position, college, draft_year,
               ROUND(AVG(forty_yard_dash), 2) AS avg_40,
               ROUND(AVG(bench_press_reps), 1) AS avg_bench,
               COUNT(*) AS players
        FROM df
        GROUP BY position, college, draft_year
        ORDER BY position, avg_40
    ) TO 'combine_summary.parquet' (FORMAT PARQUET)
""")

# Verify it wrote
summary = pd.read_parquet("combine_summary.parquet")
print(f"Exported {len(summary)} rows to combine_summary.parquet")
summary.head(10)

Exported 270 rows to combine_summary.parquet


,position,college,draft_year,avg_40,avg_bench,players
0,C,Iowa,2024,4.32,20.0,1
1,C,Stanford,2023,4.38,18.0,1
2,C,Washington,2025,4.57,20.5,2
3,C,Michigan,2025,4.64,19.0,1
4,C,BYU,2025,4.68,20.0,1
5,C,Oregon,2023,4.70,12.0,1
6,C,Colorado,2023,4.71,33.0,1
7,C,Baylor,2025,4.71,34.0,1
8,C,Notre Dame,2023,4.72,34.0,1
9,C,LSU,2025,4.73,34.0,1


## Recap

| What we did | Databricks compute used? |
|---|---|
| Listed shares & tables | No |
| Loaded 300 rows into pandas | No |
| Ran SQL with DuckDB (SELECT *, aggregations, filters) | No |
| Computed new columns in pandas | No |
| Exported to local Parquet | No |

**Total warehouse / cluster cost to the consumer: $0.00**

This is the power of the open Delta Sharing protocol — share data with anyone,
and they can consume it with standard open-source tools.